In [ ]:
import time
import torch
import torch.nn as nn
from torch.nn import functional as F
from datasets import load_dataset
from transformers import GPT2Tokenizer
import math

# Hyperparameters:

# No. of training examples processed at once.
batch_size = 8
# Context length.
train_block_size = 512
max_seq_len = 2048
# Max no of iterations.
max_iters = 1500
# No of iterations after which evaluation happens.
eval_interval = 300
# Step-size during gradient descent.
learning_rate = 3e-4
# Choosing a gpu or a cpu.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# No of batches to use while evaluating the validation loss. Since each batch has 8 training examples so in total 8 * 50 = 400 examples are being used to calculate the validation loss.
eval_iters = 50
# Dimension of each token in the vector space.
n_embd = 128
# No of attention heads required while implementing Multi-Head attention. Each head shall get n_embd / n_head = 128 / 2 = 32 dimensions.
n_head = 4
# No of transformer blocks
n_layer = 4
# randomly switches of 20% of the neurons at each step to prevent overfitting.
dropout = 0.2

# To make random operations reproducible.
torch.manual_seed(1337)

# loading the WikiText-2 dataset.
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
# loading the GPT2 tokenizer.
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT2 tokenizer by default has no padding, padding is done to make the
# sentences of the same length. We are using the "EOS"(End of Sequence) token for padding.
tokenizer.pad_token = tokenizer.eos_token

# Tokenize function:
def tokenize_function(examples):
    return tokenizer(examples["text"])

# Tokenize the whole dataset:
tokenized_datasets = dataset.map(
    tokenize_function,
    # To tokenize a batch_size no of examples at once.
    batched=True,
    # After tokenization we remove the raw text.
    remove_columns=["text"]
)

# Converts to one long token sequence
train_ids = []
for item in tokenized_datasets["train"]["input_ids"]:
    train_ids.extend(item)

val_ids = []
for item in tokenized_datasets["validation"]["input_ids"]:
    val_ids.extend(item)

# convert to tensors
train_data = torch.tensor(train_ids, dtype=torch.long)
val_data = torch.tensor(val_ids, dtype=torch.long)

# vocabulary size
vocab_size = tokenizer.vocab_size

# decode function
decode = lambda l: tokenizer.decode(l)

# data loading
def get_batch(split, seq_len):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - seq_len, (batch_size,))
    x = torch.stack([data[i:i+seq_len] for i in ix])
    y = torch.stack([data[i+1:i+seq_len+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

# To evalulate how well the model is performing on training and validation data.
@torch.no_grad() # Tells PyTorch to not compute the gradients cause during evaluation we are only measuring loss and not training.
# Computes average training and validation loss.
@torch.no_grad()
def estimate_loss(seq_len):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split, seq_len)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size, slope):
        super().__init__()
        # Q = X W_Q, K = X W_K, V = X W_V, X are the input embeddings.
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # Creates a lower triangular matrice so that the token cannot cheat by looking at the future tokens.
        self.register_buffer('tril', torch.tril(torch.ones(max_seq_len, max_seq_len)))
        self.slope = slope
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape # B = batch_size, T = train_block_size, C = embedding dimension OR head_size.
        k = self.key(x)   # (B,T,head_size)
        q = self.query(x) # (B,T,head_size)

        # Matrix Multiplication to compute the similarity between K and Q.
        # We have divided Q K^T by sqrt(d_k), d_k is the dimension of the embeddings in K.
        # Without scaling the dot products shall become huge and softmax shall become unstable.
        # Standard attention scores
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5

        # Create position indices
        positions = torch.arange(T, device=x.device)

        # Compute pairwise distances
        # shape: (T, T)
        distance = positions[:, None] - positions[None, :]

        # ALiBi bias
        alibi_bias = -self.slope * distance

        # Add bias to attention scores
        wei = wei + alibi_bias

        # Putting all the zeroes in wei as -inf so that when entered into softmax these entries shall become 0.
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        # Combines value vectors according to attention weights.
        out = wei @ v
        return out

def get_alibi_slopes(n_heads):

    def get_slopes_power_of_2(n):

        start = 2 ** (-2 ** -(math.log2(n) - 3))

        ratio = start

        return [start * ratio ** i for i in range(n)]

    # If n_heads is power of 2
    if math.log2(n_heads).is_integer():

        return torch.tensor(get_slopes_power_of_2(n_heads))

    # Otherwise handle non-power-of-2 case
    else:

        closest_power_of_2 = 2 ** math.floor(math.log2(n_heads))

        slopes_power_2 = get_slopes_power_of_2(closest_power_of_2)

        extra_slopes = get_slopes_power_of_2(2 * closest_power_of_2)

        extra_slopes = extra_slopes[0::2][:n_heads - closest_power_of_2]

        return torch.tensor(slopes_power_2 + extra_slopes)

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()

        slopes = get_alibi_slopes(num_heads)

        self.heads = nn.ModuleList([
            Head(head_size, slopes[i].item())
            for i in range(num_heads)
        ])

        self.proj = nn.Linear(head_size * num_heads, n_embd)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        out = torch.cat([h(x) for h in self.heads], dim=-1)

        out = self.dropout(self.proj(out))

        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            # Expands the total dimensions from n_embd -> 4 * n_embd. (here: 128--> 512)
            nn.Linear(n_embd, 4 * n_embd),
            # ReLU(x) = max(0, x)
            nn.ReLU(),
            # Compress back to n_embd dimensions.
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        # Layernorms, normalizes activations to stabilize training.
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # "Residual Connection", instead of replacing the old representations completely
        # the model learns the corrections and the refinements.
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # Final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights) # Initializing with random small weights(centered at 0 and std dev = 0.02) and biases.

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        x = tok_emb
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # "idx" is the full sequence generated so far.
        for _ in range(max_new_tokens):
            # "idx_cond" is the last "train_block_size" no of tokens in idx.
            idx_cond = idx[:, -max_seq_len:]
            # Get the predictions.
            logits, loss = self(idx_cond)
            # Focus only on the last time step.
            logits = logits[:, -1, :] # Becomes (B, C).
            # Apply softmax to get probabilities.
            probs = F.softmax(logits, dim=-1) # (B, C)
            # Sample from the distribution.
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # Append sampled index to the running sequence.
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = GPTLanguageModel().to(device)
# Print the number of parameters in the model.
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

# Create a PyTorch optimizer.
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

start_time = time.time()
last_eval_iter = 0

for iter in range(max_iters):

    # Every once in a while evaluate the loss on train and val sets.
    if iter % eval_interval == 0 or iter == max_iters - 1:

        if torch.cuda.is_available():
          torch.cuda.reset_peak_memory_stats()

        losses = estimate_loss(train_block_size)

        if torch.cuda.is_available():
          torch.cuda.synchronize()

        elapsed = time.time() - start_time

        iterations_done = iter - last_eval_iter
        tokens_processed = iterations_done * batch_size * train_block_size

        throughput = 0 if iterations_done == 0 else tokens_processed / elapsed

        # Perplexity
        train_ppl = torch.exp(losses['train'])
        val_ppl = torch.exp(losses['val'])

        print(f"\nstep {iter}:")
        print(f"train ppl: {train_ppl:.2f}")
        print(f"val ppl: {val_ppl:.2f}")
        print(f"throughput: {throughput:.2f} tokens/sec")

        if torch.cuda.is_available():
          memory = torch.cuda.max_memory_allocated() / 1024**3
          print(f"gpu memory: {memory:.2f} GB")

        start_time = time.time()
        last_eval_iter = iter

    # sample a batch of data
    xb, yb = get_batch('train', train_block_size)

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# Evaluating at different L_test
for test_len in [512, 1024, 2048]:

    losses = estimate_loss(test_len)

    train_ppl = torch.exp(losses['train'])
    val_ppl = torch.exp(losses['val'])

    print(f"\nLtest = {test_len}")
    print(f"Train PPL: {train_ppl:.2f}")
    print(f"Val PPL: {val_ppl:.2f}")

# ---------------- INFERENCE BENCHMARK ----------------

model.eval()

gen_tokens = 200

context = torch.randint(
    0,
    vocab_size,
    (1, train_block_size),
    device=device
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

start = time.time()

with torch.no_grad():
    model.generate(context, max_new_tokens=gen_tokens)

if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed = time.time() - start

print("\nInference Benchmark:")
print(f"Inference tokens/sec: {gen_tokens / elapsed:.2f}")
print(f"Latency per token: {elapsed / gen_tokens:.4f} sec")
# ------------------------------------------------------

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))
#open('more.txt', 'w').write(decode(m.generate(context, max_new_tokens=10000)[0].tolist()))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Map:   0%|          | 0/4358 [00:00<?, ? examples/s]

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

Map:   0%|          | 0/3760 [00:00<?, ? examples/s]

13.707857 M parameters

step 0:
train ppl: 51309.67
val ppl: 51368.08
throughput: 0.00 tokens/sec
gpu memory: 2.62 GB

step 300:
train ppl: 769.14
val ppl: 814.50
throughput: 33216.90 tokens/sec
gpu memory: 3.55 GB

step 600:
train ppl: 440.04
val ppl: 501.28
throughput: 29170.62 tokens/sec
gpu memory: 3.55 GB

step 900:
train ppl: 327.24
val ppl: 410.16
throughput: 29210.68 tokens/sec
gpu memory: 3.55 GB

step 1200:
train ppl: 261.87
val ppl: 349.24
throughput: 27161.07 tokens/sec
gpu memory: 3.55 GB

step 1499:
train ppl: 222.28
val ppl: 303.31
throughput: 27677.17 tokens/sec
gpu memory: 3.55 GB

Ltest = 512
Train PPL: 209.50
Val PPL: 317.57

Ltest = 1024
Train PPL: 215.89
Val PPL: 313.40

Ltest = 2048
Train PPL: 209.13
Val PPL: 318.05

Inference Benchmark:
Inference tokens/sec: 103.17
Latency per token: 0.0097 sec
! are 400 metres ( 20 months to mock days ) . The tale conMA Ment pour and fighting , the remains of the French era and roof is appointed as old heavy government and promp